## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [1]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [2]:
# Always remember to do this!
load_dotenv(override=True)

True

In [3]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
DeepSeek API Key exists and begins sk-
Groq API Key exists and begins gsk_


In [4]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [5]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation.'}]

In [6]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


You are the sole decision-maker responsible for the immediate crisis response on an island of 10,000 inhabitants where a volcano is forecast to erupt within 72 hours (with a 30% chance of occurring within 48 hours and a 20% chance of being delayed to 96 hours). You have the following resources and constraints:
- Transportation: 5 buses (50 seats each), 10 ambulances (4 seats each); fuel sufficient for two round trips of the buses and four round trips of the ambulances.
- Shelter and medical: a temporary shelter 3 km inland that holds 3,000 people with two weeks of basic supplies; a hospital with 200 beds and limited ICU; field medical teams that can expand capacity by 50 beds but will take 18 hours to set up; a dam downstream that, if unattended, has a 60% chance of failing within 48 hours and would flood lowlands killing ~2,000 people.
- Evacuation infrastructure: a military airstrip that can evacuate 200 people per sortie but requires a team of five trained operators to run; currentl

In [7]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Note - update since the videos

I've updated the model names to use the latest models below, like GPT 5 and Claude Sonnet 4.5. It's worth noting that these models can be quite slow - like 1-2 minutes - but they do a great job! Feel free to switch them for faster models if you'd prefer, like the ones I use in the video.

In [8]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-4.1-mini"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

**Context Summary:**  
Volcano expected to erupt within 72h (30% chance <48h, 50% chance 48-72h, 20% chance 72-96h). Dam downstream likely to fail (60% chance within 48h if unattended), flooding lowlands and killing ~2,000. Population 10,000 with vulnerable groups and infrastructure constraints. Limited transport, shelter, and medical resources. Social/legal constraints prohibit nationality-based prioritization, demand protection of elderly/children, and require guarding incarcerated population. Airstrip limited by trained operator availability.

---

## 1. Assumptions and Objective Function

### Assumptions:
- **Eruption timing probabilities:**  
   - <48h: 30%  
   - 48-72h: 50%  
   - 72-96h: 20%  
- **Dam breach probability if unattended:** 60% within 48h, causing ~2,000 deaths. Dam can only be maintained by trained workers.  
- **Evacuation transport capacities fixed:**  
   - Buses: 5 × 50 = 250 seats; 2 round trips → 500 seats total  
   - Ambulances: 10 × 4 = 40 seats; 4 round trips → 160 seats total  
   - Airstrip: 200 seats/sortie; 2 trained operators now → at most 2 simultaneous sorties; max ~400 evacuees assuming multiple sorties  
- **Shelter capacity:** 3,000 people, 3km inland, with 2 weeks supplies.  
- **Hospital capacity:** 200 beds + 50 expansion (after 18h set-up).  
- **Population grouping:**  
   - 1,200 children under 12  
   - 800 elderly 65+  
   - 150 current inpatients (20 ICU)  
   - 400 critical infrastructure workers (water, power, dam)  
   - 300 tourists (no prioritization by nationality)  
   - 200 incarcerated (low-security, unrest)  
- **Legal constraints:** no nationality prioritization, must protect elderly and children visibly.  
- **Social constraints:** incarcerates must be guarded to avoid escalation. Social media panic is ongoing and growing.

### Objective function:
**Minimize expected total fatalities over 7 days, weighted by vulnerability and system criticality, plus minimize risk of catastrophic infrastructure loss and social collapse.**

Formalization:  
Minimize  
E[Fatalities] + λ × (Expected deaths from system failures + social instability impact),  
where fatalities weigh vulnerable groups higher (children and elderly given higher weight by factor 2), and λ balances human loss vs infrastructure collapse risk (chosen lambda to prioritize human lives but give serious weight to system failures).

---

## 2. Prioritized 72-Hour Operational Plan

| **Time (h)** | **Action** | **Details/Notes** |
|----------------|-------------|-------------------|
| **Hour 0 (Now)** | **Establish command & communication; rapid reassessment teams to dam site and social teams assessing incarceration risk** | Begin training 3 more operators for airstrip; prioritize dam workers’ availability; start calming communications handling panic. |  
| **Hour 0–6** | **Evacuate highest risk + critical infrastructure personnel + vulnerable groups (priority: dam workers, ICU patients, elderly, children)** |  
- Ambulances: 160 seats × 4 trips = 640 seats total → move ICU (20), other inpatients (130), elderly with chronic conditions (800, constrained by seats, partial evacuation), children (start moving them), incr. some tourists |  
- Buses: 250 seats × 2 trips (500 seats total) → elderly + children + tourists to shelter 3 km inland |  
- Keep a security team to guard incarcerated (no evacuation possible) |  
| **Hour 6–18** | **Set up field medical expansion (+50 beds); continue evacuation of vulnerable plus tourists + non-critical personnel** |  
- Ambulances/buses second trips continue. Remaining elderly/children, plus tourists to shelter. Total shelter population capped at 3,000. Shelter priority: children + elderly + tourists + some workers |  
- Prioritize keeping water/power/dam workers on-site or near dam |  
- First airstrip sortie evacuates highest medical risk (ICU patients, serious chronic elderly) if trained operators ready (limit 2 sorties max before new ops arrive) |  
| **Hour 18–24** | **Deploy stable patients/remaining vulnerable to shelter and hospital (expanded beds). Continue evacuation flights if possible.** |  
- New operators finish training around hour 24; airlift of additional operators can be started if critical (36h minimum) |  
| **Hour 24–48** | **Monitor volcano & dam; begin immediate dam maintenance shift rotations with critical workers to prevent failure; prioritize protecting dam workers in shelter/housing near dam** |  
- Airstrip operations increase to evacuate tourists and non-vulnerable willing to evacuate |  
- Continue sheltering elderly, children, tourists, critical workers |  
- Consider limited use of ambulances/buses for shuttling manpower |  
| **Hour 48–72** | **Prepare for eruption response & possible evacuation continuation depending on eruption timing & dam stability** |  
- Rotate dam maintenance team keeping at least 50 workers present to avoid dam failure |  
- Use airstrip for rapid evacuation of remaining vulnerable if eruption earlier than 72h |  
- Monitor incarceration carefully; deploy calming/social services to prevent unrest |  

---

### Transport and Fuel allocation:

- Helicopters unavailable.  
- Fuel usage:  
   - Ambulances: 4 round trips × 10 ambulances × 4 seats = 160 capacity; prioritize ICU/inpatients first, then elderly/children.  
   - Buses: 2 round trips × 5 buses × 50 seats = 500 capacity; prioritize children, elderly, tourists to shelter.  
- Airstrip: Initially 2 operators → max 2 sorties (200 ppl each) before 24h mark. Train 3 more operators in parallel to increase sortie rate after 24h.

---

### Shelter and Medical Capacity:

- Shelter: 3,000 capacity prioritized for children/elderly + tourists → meets demand for these vulnerable groups + some workers.  
- Hospital: 200 beds + 50 expansion set up by 18h → host critical inpatients + severe cases from evacuation.  
- Field medical teams prioritized for rapid set-up.

---

### Social/legal constraints handling:

- Guard incarcerated continuously (minimum 20 guards from workers and security staff on-site) to prevent escalation during evacuation phases.  
- Transparency and visible protection of children and elderly to satisfy political pressure and reduce panic.  
- No nationality-based prioritization: tourists evacuated according to seat availability and risk, mixed with locals (priority being vulnerability).  
- Media team to counter misinformation actively starting immediately.

---

## 3. Quantified Trade-offs and Expected Outcomes

| Scenario | Eruption timing | Dam failure | Intervention result | Expected fatalities (estimates) |
|-----------|-----------------|-------------|---------------------|---------------------------------|
| **Best case** | Eruption delayed (≥72h) | Dam holds | Full evacuation for vulnerable via bus/ambulance/airstrip complete; dam maintained; incarceration stable | ~50 deaths (hospital/evacuation medical failures + possible panic incidents) |
| **Most likely** | Eruption 48-72h (50%) | Dam holds/fails (60% chance dam fails if abandoned) | Partial evacuation achieved, dam workers rotate in shifts; dam narrowly held; limited airstrip sorties before operators ready | ~300-500 deaths (ICU patients + some elderly not evacuated + possible dam breach victims if dam crew fail) |
| **Worst case** | Eruption <48h (30%) | Dam fails | Evacuation incomplete; dam failure causes flooding kills ~2,000; eruption kills 300+ from un-evacuated zones; unrest escalates | 2,300+ deaths (dam flood + eruption + social collapse-related deaths) |

**Trade-offs:**  
- Prioritizing dam worker retention decreases dam failure risk and reduces expected deaths by up to 2,000, a massive reduction justifying keeping significant workers onsite.  
- Prioritizing children and elderly first complies with political/legal/social needs but limits number of able-bodied workers evacuated (critical for dam & utilities). Balance is essential.  
- Airstrip capacity constrained by trained operators: training more increases evacuation rate post 24h, justifying training focus.

---

## 4. Adaptive Decision Updates

**New information would trigger updates in:**

- **Eruption confirmed imminent (<24h):**  
   - Immediately stop training operators, instead run max airstrip sorties with current operators.  
   - Prioritize full evacuation of vulnerable and those near volcanic risk zones, suspend dam maintenance if safety endangered.  
   - Possibly start rapid shelter expansion or secondary shelter arrangements.  

- **Dam indicates increased instability (beyond 60% failure risk):**  
   - Increase critical dam workers onsite, suspend evacuations of these workers.  
   - Evacuate all non-dam workers ASAP.  
   - Initiate local flood warning systems and pre-emptive lowlands evacuation if possible.  

- **Additional trained operators arrive sooner/later:**  
   - Sooner: ramp up airstrip evacuation pace; move more vulnerable out by air.  
   - Later: extend more evacuation by road/ambulance; ramp up shelter capacity absorption.

- **Social unrest among incarcerated is escalating:**  
   - Reinforce guard presence, stabilize situation via negotiation; avoid mass transfers unless absolutely necessary.  
   - Increase communication and transparency to reduce unrest.

---

## 5. Three Plausible Failure Modes, Consequences, and Mitigations

| Failure Mode | Consequences | Mitigation |
|--------------|--------------|------------|
| **1. Dam failure due to insufficient critical workers onsite or mishandling** | Flood killing ~2,000, massive infrastructure loss, causing chaotic additional deaths | Assign clear rotation schedules, maintain communication with dam workers, provide safe onsite shelter and food to incentivize staying; establish backup technical support route; monitor dam closely |
| **2. Evacuation transport logistics failure (vehicle breakdown, fuel misallocation)** | Reduced evacuee movement, leaving vulnerable exposed; traffic bottlenecks add panic and injuries | Pre-departure vehicle check, contingency refuel plans, designated alternate routes, staged evacuation timing; reserve emergency fuel for priority transports |
| **3. Social collapse due to incarceration unrest or misinformation-driven panic** | Riot, facility breach causing injuries/deaths; mass panic disrupting orderly evacuation; trust erosion | Deploy social services, visible law enforcement presence; clear communication strategy targeting social media misinformation; regular updates to community leaders; engage mental health teams |

---

## Ethical and Legal Decision Influences

- **No nationality-based prioritization:** tourists treated equally with locals; evacuation ordered by vulnerability and needs only, satisfying legal constraints.  
- **Visible protection of elderly and children:** politically and ethically prioritized for evacuation and sheltering first to comply with local demands and reduce vulnerable fatalities.  
- **Incarcerated must not be abandoned:** guarded to avoid social unrest and violent outbreaks, despite added resource burden.  
- **Transparency and non-discrimination:** communication and action respect human rights, reducing panic and social instability.  
- **Balancing critical infrastructure protection:** retaining dam workers onsite prioritized to minimize catastrophic deaths, ethically justified as collective survival strategy.

---

# **Summary Plan**

- **Immediate evacuation with ambulances + buses of highest-risk groups to shelter/hospital.**  
- **Retain critical dam and infrastructure workers onsite in shifts to prevent dam breach and maintain utilities.**  
- **Train additional airstrip operators ASAP; deploy air evacuation continuously post 24h, prioritizing medical cases and vulnerable.**  
- **Maintain incarceration guards to prevent collapse of security.**  
- **Proactive communication strategy to combat panic and misinformation.**  
- **Constant monitoring with decision points for plan adjustment upon new eruption or dam status data.**

---

By executing this plan, we optimally reduce expected fatalities (weighted for vulnerability) and system collapse risk, balancing legal/social constraints and physical resource limits.

In [9]:
# Anthropic has a slightly different API, and Max Tokens is required

model_name = "claude-sonnet-4-5"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

# VOLCANIC ERUPTION CRISIS RESPONSE PLAN
## Island Population: 10,000 | Timeline: 72 hours

---

## 1. ASSUMPTIONS AND OBJECTIVE FUNCTION

### Primary Assumptions:
1. **Eruption modeling**: 30% chance within 48h, 50% within 72h (baseline), 20% delayed to 96h
2. **Dam criticality**: 60% failure probability within 48h if unattended; failure kills ~2,000 in lowlands (assumed evenly distributed population)
3. **Transport times**: Bus round-trip = 6 hours (3km each way + loading/unloading); ambulance round-trip = 3 hours; air sortie = 4 hours
4. **Shelter capacity**: 3,000 at inland shelter; overflow requires field medical setup or mainland evacuation
5. **Personnel assumptions**: Dam requires minimum 4 operators continuously; airstrip requires 5 trained operators for full capacity; training in-situ takes 30 hours per person
6. **ICU patients**: Cannot survive bus transport; require ambulance or air evacuation
7. **Social stability**: Leaving incarcerated population unguarded >12 hours = 70% chance of breach/violence
8. **Volcanic hazard zone**: Entire island at risk; inland shelter provides pyroclastic flow protection but not total safety

### Objective Function (Lexicographic Priority):
**Primary**: Minimize expected deaths over 7 days  
**Secondary**: Prevent catastrophic infrastructure collapse (dam failure)  
**Tertiary**: Maintain social order and equitable treatment per legal constraints  

**Expected death calculation**:  
E[deaths] = P(eruption) × (vulnerable population not evacuated) + P(dam_failure) × 2,000 + P(social_collapse) × estimated casualties

---

## 2. PRIORITIZED 72-HOUR OPERATIONAL PLAN

### HOUR 0-6: Critical Infrastructure & Immediate Triage

**Immediate Actions (Hour 0-2):**

1. **Dam stabilization (CRITICAL)**
   - Deploy 4 dam operators immediately with security escort
   - Allocate 1 ambulance permanently for dam team rotation (reserves 1 of 4 round-trips)
   - **Rationale**: Prevents certain 2,000 deaths; 60% failure risk unacceptable
   
2. **Airstrip activation**
   - Airlift 3 additional trained operators (ETA: 24-36h → prioritize for 24h arrival)
   - 2 current operators begin 24/7 operations at reduced capacity (100 people/sortie vs 200)
   - **Rationale**: Creates high-capacity evacuation route for most vulnerable
   
3. **Communications & social control**
   - Establish official information channel; deploy police/community leaders to counter misinformation
   - Announce transparent evacuation priority (vulnerability-based, not nationality)
   - Deploy 8 guards to incarcerated facility with keys to cell release system if facility must be abandoned
   
4. **Field medical team deployment**
   - Begin 18-hour setup at inland shelter (completion: Hour 18)
   - **Adds 50 beds for medical needs**

**Transport Operations (Hour 0-6):**

| Time | Asset | Payload | Destination | Rationale |
|------|-------|---------|-------------|-----------|
| H0-3 | 3 ambulances | 20 ICU patients (4×5 trips) | Mainland via airstrip staging | Cannot survive bus transport; highest mortality risk |
| H0-6 | 5 buses (Trip 1/2) | 250 priority patients + 100 elderly with medical needs | Inland shelter | Use first of two bus round-trips for medically fragile |
| H0-4 | Air (reduced) | 100 elderly/disabled | Mainland | Begin continuous air ops with 2 operators |

**Running totals by Hour 6:**
- Evacuated: 20 ICU (mainland) + 350 medical/elderly (shelter) + 100 elderly (mainland) = 470
- Dam: Secured (4 operators in place)
- Remaining: 9,530

---

### HOUR 6

In [10]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

This plan addresses the immediate crisis on the island, balancing the high probability of a dam failure with the impending volcanic eruption, while navigating severe resource constraints and social pressures.

---

### 1) Assumptions and Primary Objective Function

**Primary Objective Function:** Minimize expected fatalities over the next seven days, weighted to prioritize the most vulnerable populations (children under 12, elderly 65+, inpatients, and critical workers whose absence would cause widespread death). Secondary objectives are to prevent catastrophic infrastructure failure (specifically the dam) and mitigate the risk of social collapse.

**Key Assumptions:**

*   **Volcano Probability:**
    *   Probability of eruption within 48 hours: 30%
    *   Probability of eruption between 48 and 72 hours: 25% (derived by assuming uniform distribution for the 50% chance of eruption between 48h and 96h, and 20% chance of delay beyond 96h, so 100% - 30% - 20% = 50% between 48-96h; thus 50%/2 = 25% for 48-72h).
    *   Therefore, the probability of eruption within the 72-hour planning window is 30% + 25% = 55%.
*   **Dam Maintenance:** 100 of the 400 critical workers are required *at the dam* to prevent its failure. The remaining 300 are crucial for other essential services (water, power) and are encouraged to remain, but their safety is a concern.
*   **Ground Transport Turnaround:** Buses and ambulances can complete a round trip to the 3km shelter within approximately 30-45 minutes. Fuel is the limiting factor, not time.
*   **Airstrip Operator Arrival:** The 3 additional operators are assumed to arrive or be trained within 30 hours (mid-point of 24-36h window). With 5 operators, the airstrip can manage one sortie every 2 hours, evacuating 200 people per sortie.
*   **Shelter Safety:** The temporary shelter 3km inland is considered safe from direct lava flow/pyroclastic density currents, but may experience heavy ashfall and associated disruptions.
*   **Incarcerated Population Security:** 10 dedicated security personnel are needed to manage the 200 incarcerated individuals, moving with them to a secure location. These personnel are drawn from the general population.
*   **Voluntary Compliance:** A reasonable level of public cooperation with evacuation orders is assumed, though panic and misinformation are acknowledged risks.
*   **External Aid:** No additional ground transport, medical supplies (beyond field teams), or security personnel are assumed to arrive within 72 hours, other than the requested airstrip operators.

---

### 2) Prioritized 72-Hour Operational Plan

**Phase 1: 0-24 Hours - Immediate Threats & Preparation**

*   **Objective:** Secure dam, initiate emergency medical care and vulnerable group sheltering, prepare airstrip for full operation, establish control.

    *   **T=0-2h: Dam & Critical Personnel Security**
        *   **Action:** Immediately dispatch 100 critical dam maintenance workers to the dam site. Provide emergency rations, communications, and safety instructions.
        *   **Rationale:** Mitigates the 60% chance of 2,000 deaths from dam failure within 48 hours, the most immediate and largest probable catastrophe.
        *   **Action:** Contact the remaining 300 water/power workers; instruct them to prepare for family evacuation and stand by for essential duties.
        *   **Action:** Initiate urgent communication with external authorities to prioritize the airlift/training of 3 additional airstrip operators, emphasizing the 24-36 hour requirement.
    *   **T=0-6h: Critical Medical & Vulnerable Ground Evacuation (to Shelter)**
        *   **Priority 1: Hospital Inpatients**
            *   **Who:** 150 inpatients (including 20 ICU-level).
            *   **Transport:** 10 ambulances (4 seats each). Each ambulance performs 4 round trips. Total capacity: 10 * 4 * 4 = 160 people.
            *   **Where:** Hospital to Temporary Shelter (3km inland).
            *   **Outcome:** All 150 inpatients transported safely. Immediately begin setting up field medical teams at the shelter.
            *   **Fuel Used:** All ambulance fuel.
        *   **Priority 2: Most Vulnerable Elderly & Children (initial wave)**
            *   **Who:** 250 most frail elderly and youngest children.
            *   **Transport:** 5 buses (50 seats each). Each bus performs 2 round trips. Total capacity: 5 * 50 * 2 = 500 people.
            *   **Where:** Designated community points to Temporary Shelter.
            *   **Outcome:** 250 high-risk elderly/children transported.
            *   **Fuel Used:** All bus fuel.
    *   **T=0-12h: Airstrip Initial Activation & Field Medical Setup**
        *   **Action:** Activate the military airstrip with the 2 existing trained operators.
        *   **Who:** First 200 people for evacuation via airstrip will be critically ill family members of dam workers (if not in shelter), or other highly vulnerable individuals that couldn't fit in initial ground transport. A single sortie can be managed, but slowly.
        *   **Action:** Commence setup of field medical teams at the temporary shelter to expand capacity by 50 beds (expected operational at T+18h).
    *   **T=0-24h: Information Management & Security Assessment**
        *   **Action:** Establish a central emergency information hub. Disseminate clear, frequent, and factual updates via all media (radio, local loudspeakers, verified social media channels) to counter panic and misinformation.
        *   **Action:** Reinforce security at the incarcerated facility; prepare for their eventual secure relocation.

**Phase 2: 24-48 Hours - Airstrip Ramp-Up & Mass Evacuation**

*   **Objective:** Maximize external evacuation; consolidate shelter; manage incarcerated population.

    *   **T=24-36h: Airstrip Full Operations & Continued Evacuation**
        *   **Action:** The 3 additional airstrip operators arrive/are trained. Airstrip is now fully operational (5 operators).
        *   **Airstrip Operations:** Aim for 1 sortie every 2 hours (200 people/sortie). Over 12 hours, this is 6 sorties = 1,200 people.
        *   **Prioritization for Airstrip:**
            1.  Remaining 550 elderly (800 total - 250 sheltered)
            2.  Remaining 950 children (1,200 total - 250 sheltered)
            3.  300 non-native tourists
            4.  Families of critical workers (dam/infrastructure)
    *   **T=24-48h: Shelter Consolidation & Medical Expansion**
        *   **Action:** The 50 expanded medical beds at the temporary shelter are now operational, supporting the inpatients and any new arrivals.
        *   **Action:** Continue encouraging self-evacuation of general population to the temporary shelter, guiding them to available space (capacity for ~2,400 more).
    *   **T=24-48h: Incarcerated Population Relocation**
        *   **Action:** Move the 200 incarcerated individuals, accompanied by 10 security personnel, to a designated secure section within the temporary shelter or a separate secure inland facility.
        *   **Rationale:** Ensures their safety from primary threats and mitigates unrest by providing a secure, managed environment, avoiding mixing with general population evacuees.

**Phase 3: 48-72 Hours - Final Evacuation Push & Contingency Planning**

*   **Objective:** Maximize remaining external evacuation; ensure dam stability; prepare for eruption; provide last-minute guidance.

    *   **T=48-72h: Maximize Airstrip Evacuation**
        *   **Action:** Continue airstrip operations at full capacity. Aim for 1 sortie every 2 hours. Over 24 hours, this is 12 sorties = 2,400 people.
        *   **Prioritization:** Remaining families of critical workers, and general population.
    *   **T=48-72h: Dam & Infrastructure**
        *   **Action:** Maintain 100 workers at the dam, with continuous monitoring.
        *   **Action:** For the remaining 300 water/power workers, offer them last-minute evacuation opportunities via airstrip if available, or instruct them on pre-identified safe zones if they choose to remain.
    *   **T=48-72h: Final Shelter & Public Information**
        *   **Action:** Consolidate all sheltered populations. Ensure supplies are distributed and medical aid is available.
        *   **Action:** Issue final evacuation warnings for those who choose to remain or are beyond reach. Provide clear instructions on what to do during an eruption (e.g., seek sturdy shelter, cover mouth/nose, stay indoors). Prepare for post-eruption search and rescue.

---

### 3) Quantify Trade-offs and Expected Outcomes

**Population Accounting (after 72 hours):**

*   **Evacuated via Airstrip (T=36h to T=72h):** 1,200 (Phase 2) + 2,400 (Phase 3) = **3,600 people** (including remaining elderly, children, tourists, and general population).
*   **Sheltered (at 3km inland shelter):**
    *   150 Inpatients + 250 initial Elderly/Children = 400
    *   200 Incarcerated + 10 Security = 210
    *   *Total minimum committed to shelter:* 610 people. (Remaining shelter capacity: 3,000 - 610 = 2,390 for general population who self-evacuate or use local transport).
*   **Critical Workers Remaining on Island:**
    *   100 Dam Workers (at dam)
    *   300 Other Infrastructure Workers (water/power, some may evacuate, some will stay)
    *   *Assume 400 critical workers remain initially for calculation of risk.*
*   **Total Accounted For (Moved to Safety or Critical Roles):** 3,600 (airstrip) + 610 (sheltered) + 400 (critical workers) = **4,610 people.**
*   **Remaining on Island (At Risk):** 10,000 (total population) - 4,610 = **5,390 people.**
    *   Of these, up to 2,390 can find space in the temporary shelter if they can reach it.
    *   Remaining true at-risk population who couldn't/wouldn't evacuate or shelter: 5,390 - 2,390 = **3,000 people.**

**Expected Fatalities Under Plan:**

*   **Dam Failure:**
    *   **Trade-off:** 100 workers at risk.
    *   **Outcome:** By immediately deploying 100 workers, the 60% chance of 2,000 deaths from dam failure is effectively mitigated to **0 expected deaths** from this specific hazard.
*   **Volcano Eruption (P=55% within 72h):**
    *   **Averted Deaths:** Approximately 2,450 high-vulnerability individuals (150 Inpatients, 800 Elderly, 1200 Children, 300 Tourists) are evacuated or safely sheltered, largely averting fatalities for these groups.
    *   **Critical Workers (400):** These individuals choose to remain for critical functions. If eruption occurs, assume a 50% fatality rate for those in high-risk areas.
        *   Expected deaths = 400 * 0.50 (fatality rate) * 0.55 (P_eruption) = **110 deaths.**
    *   **Remaining At-Risk General Population (~3,000 people):** This group is either in a high-risk zone or could not reach safety. Assume 40% are in a direct high-risk zone and, if an eruption occurs, 70% of those in the zone will die.
        *   Expected deaths = 3,000 * 0.40 (in high-risk zone) * 0.70 (fatality rate) * 0.55 (P_eruption) = **462 deaths.**
    *   **Total Expected Fatalities:** 110 (critical workers) + 462 (general population) = **572 deaths.**

**Worst-Case Scenario (Under Plan):**
*   Eruption occurs at T=0 (P=30%). Dam fails before workers are fully effective (highly unlikely with immediate dispatch, but possible). Airstrip has only 2 operators for 36 hours.
*   Dam Failure: 2,000 deaths.
*   Limited Vulnerable Moved: 150 inpatients + 250 elderly/children = 400.
*   Remaining Vulnerable (2450 - 400 = 2050) die due to eruption: 2,050 deaths.
*   General Population (10,000 - 400 vul - 200 incarc - 400 workers = 9,000) largely dies (e.g., 50%): 4,500 deaths.
*   **Worst-Case Total Deaths (Early Eruption, Dam Fails): ~8,550 deaths.** (This represents a catastrophic failure of early response measures.)

**Best-Case Scenario:**
*   Eruption delayed beyond 96 hours (P=20%).
*   Dam remains secure.
*   All 4,610 people successfully evacuated or sheltered.
*   The remaining 5,390 people (including the 2,390 who self-shelter) find safe zones or evacuate through other means (e.g., private boats, extended airstrip operations).
*   **Best-Case Total Deaths: Approaching 0 deaths.**

---

### 4) Updating Decisions as New Information Arrives

This plan is dynamic and will be adjusted based on real-time data:

*   **Eruption Confirmed in 24 Hours (P=30% scenario becomes 100% certainty):**
    *   **Change:** Immediately halt all non-essential operations. Prioritize a desperate push for airstrip evacuation (even with limited operators, attempt more sorties). Immediately secure families of dam/critical workers if not already done. Issue an island-wide "take cover" directive, ordering all remaining general population to immediately move to the temporary shelter or nearest designated sturdy structure/high ground. Incarcerated population moved immediately.
    *   **Threshold:** Confirmation from geological monitoring (e.g., sustained high-amplitude harmonic tremor, verified lava dome growth, phreatic explosions) indicating an eruption is hours away.
*   **Dam Shows Increased Instability (e.g., major cracks, rapid water level changes, P > 60% for failure):**
    *   **Change:** Double the number of workers at the dam (diverting from other infrastructure teams if necessary). Provide emergency local transport for dam workers' immediate families to the shelter. Prepare for a controlled breach/release if total failure is unavoidable, to mitigate peak flood levels, even if it causes minor flooding.
    *   **Threshold:** Engineering assessment indicates a high probability (>80%) of failure within 6 hours, or visible, rapid structural degradation.
*   **Additional Trained Operators Arrive Sooner (e.g., at T+12h instead of T+30h):**
    *   **Change:** Immediately ramp up airstrip operations to full capacity. This allows more vulnerable groups to be evacuated earlier, increasing overall safety. More general population can also be evacuated within the 72-hour window.
    *   **Threshold:** Verified arrival time from external agencies.
*   **Additional Trained Operators Delayed (e.g., > 48 hours):**
    *   **Change:** Adjust airstrip capacity downwards significantly. Focus the few available sorties on only the most critically vulnerable (e.g., ICU patients, most frail elderly). Reallocate efforts to further enhance temporary shelter capacity, expand secondary safe zones, and prepare for a longer stay for a larger on-island population.
    *   **Threshold:** Official communication confirming a delay of more than 12 hours from the anticipated arrival.
*   **Shelter Reaches Capacity (3,000 people):**
    *   **Change:** Immediately designate and publicize secondary, less-equipped safe zones (e.g., high ground outside hazard zones, other sturdy buildings). Direct remaining population there. Prioritize distribution of basic supplies (water, first aid) to these new zones.
    *   **Threshold:** Actual occupancy numbers reaching 2,800.

---

### 5) Three Plausible Failure Modes and Mitigations

1.  **Failure Mode: Widespread Social Collapse and Panic (Social/Informational)**
    *   **Likely Consequences:** Uncontrolled mass movement, blocking evacuation routes, stampedes, looting, breakdown of order, self-evacuation into unsafe areas, inability to implement planned evacuations. Unrest among the incarcerated population escalating into a security crisis.
    *   **Mitigations:**
        *   **Immediate:** Establish a unified, trusted information channel (e.g., "Crisis Command Radio") for continuous, factual, calm updates. Use local community leaders and trusted figures to disseminate information and reassure the public.
        *   **Ongoing:** Deploy visible security personnel (police, available military) at critical evacuation points, the shelter, and the incarcerated facility to maintain order and provide reassurance. Implement a "buddy system" to encourage mutual aid. For the incarcerated population, clear communication about their relocation plan and access to basic needs will be critical, alongside dedicated security.
2.  **Failure Mode: Airstrip Operational Failure/Severe Delays (Operational)**
    *   **Likely Consequences:** Significantly reduced off-island evacuation capacity, leaving thousands stranded and increasing the burden on the limited inland shelter. This would directly increase expected fatalities and exacerbate public panic and frustration.
    *   **Mitigations:**
        *   **Immediate:** Prioritize the request for additional airstrip operators above all other external aid. Conduct immediate cross-training of local personnel on basic airstrip support tasks (e.g., ground guiding, basic security, logistics) to build redundancy.
        *   **Ongoing:** Establish clear, disciplined queuing and boarding procedures at the airstrip, with medical personnel to handle emergencies. Develop contingency plans for aircraft delays (e.g., providing temporary shelter/refreshments) and potential re-routing of evacuees to the inland shelter if external evacuation becomes unfeasible. Conserve residual ground transport fuel for critical transfers to/from the airstrip if needed.
3.  **Failure Mode: Dam Failure Despite Intervention (Operational/Environmental)**
    *   **Likely Consequences:** Catastrophic flooding of lowlands, resulting in ~2,000 immediate fatalities and widespread destruction of critical infrastructure (power, water systems, housing), severely hampering post-eruption recovery efforts and potentially causing widespread disease outbreaks.
    *   **Mitigations:**
        *   **Immediate:** Beyond dispatching the 100 workers, establish real-time, redundant communication links to the dam site. Equip workers with specialized tools and engineering support where possible. Pre-identify secondary, high-ground safe zones specifically for lowland residents, for immediate evacuation if dam failure becomes imminent.
        *   **Ongoing:** Continuously monitor dam stability with engineering experts if available. Develop a rapid "fail-safe" plan: if dam integrity degrades beyond repair, issue an *immediate and specific* evacuation order for all lowland residents (even if it conflicts with volcanic eruption advice at that exact moment) to designated high ground, prioritizing flood survival. If possible, consider a controlled, strategic release of water to reduce the peak flood surge, if this option is feasible without exacerbating the problem.

In [12]:
# Updated with the latest Open Source model from OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


**1.  ASSUMPTIONS & OBJECTIVE FUNCTION**

| # | Assumption (made explicit) |
|---|----------------------------|
|A1|The “temporary shelter 3 km inland” is outside the direct path of lava, pyro‑clastic flow and the tsunami‑type surge that could follow the eruption, but it will be exposed to ash and possible flood from the downstream dam. |
|A2|All people who reach the shelter are **alive** at the moment they arrive; subsequent deaths are only from secondary hazards (ash inhalation, flood, lack of medical care). |
|A3|Fuel figures are *total* for the whole fleet: **5 buses** can each make **two round‑trips** (four one‑way legs) → 5 × 50 × 2 = 500 person‑seats outbound. **10 ambulances** can each make **four round‑trips** → 10 × 4 × 4 = 160 person‑seats outbound. |
|A4|The airstrip can handle **one aircraft** that carries **200 people per sortie**. A sortie (load → take‑off → land → refuel) takes ~8 h, so in a 72‑h window a maximum of **three** sorties is realistic if the required crew (5 operators) is assembled by hour 24. |
|A5|All 150 in‑patients (including the 20 ICU cases) are *medically* stable enough to travel in an ambulance for ≤ 2 h (the distance to the shelter). |
|A6|The “workers who maintain water, power and dam systems” (400 people) are **essential**: at least **200** must stay on the island to keep basic services and to try to prevent dam failure. |
|A7|The low‑security incarcerated group (200 people) must be **guarded** at all times; a contingent of **50 guards** (drawn from the worker pool) will be assigned to them and will stay on the island. |
|A8|Legal constraint: **no** prioritisation by nationality.  All tourists are treated as any other civilian. |
|A9|Political/social constraint: visible protection of **children (< 12 y)** and **elderly (≥ 65 y)** is required; the plan therefore places them at the front of every evacuation wave. |
|A10|Probability model for the eruption (based on the forecast):  <br>• 48 h window – 30 % chance  <br>• 72 h window – 50 % chance (30 % + 20 %)  <br>• 96 h window – 20 % chance (delayed) |
|A11|If the downstream dam fails, the low‑lying floodplain will be inundated **≈ 2 000 people** (mostly non‑essential residents).  The dam‑failure probability is 60 % if no additional action is taken; each **extra 10 workers** deployed on the dam for 24 h cuts that probability by **≈ 5 %** (linear approximation). |
|A12|Casualty risk per person left on the island at the moment of eruption (no shelter):  <br>• Children – 40 %  <br>• Elderly – 50 %  <br>• In‑patients/ICU – 70 % (if not already in the shelter)  <br>• General adult – 20 %  <br>These figures are *conditional* on an eruption occurring in the 48‑h window; they drop to 10 %/15 %/30 %/5 % if the eruption is delayed to 96 h (people have more time to self‑protect). |
|A13|If the dam fails **after** the shelter is filled, the 2 000 people who would be flooded are assumed to be **outside the shelter** (the shelter sits on higher ground).  The dam‑failure risk therefore adds an *independent* expected death toll of 0.6 × 2 000 = 1 200 if we do **nothing** about the dam. |
|A14|All calculations are **expected‑value** (probability × fatalities).  “Best‑case” = eruption at 96 h, dam holds; “worst‑case” = eruption at 48 h, dam fails. |

**Primary objective:** **Minimise the expected number of deaths over the 7‑day horizon**, with a secondary weighting that gives **twice the weight to children and the elderly** (reflecting the political demand for visible protection).  The optimisation function therefore is  

\[
\text{Minimise } E[D]=\sum_{g} w_g \, P_{\text{eruption}} \, p_{g}\, N_{g}^{\text{left}}
\;+\; P_{\text{dam‑fail}} \times 2\,000
\]

where \(w_g=2\) for children and elderly, \(w_g=1\) for all other groups, \(p_g\) is the conditional mortality risk from A12, and \(N_{g}^{\text{left}}\) is the number of persons of group *g* who are **not** in the shelter or evacuated off‑island when the eruption occurs.

---

**2.  PRIORITIZED 72‑HOUR OPERATIONAL SCHEDULE**

All times are measured from **T = 0 h** (the moment the plan is launched).

| Time‑block | Action (who, how, where) | Resources used | Rationale |
|------------|--------------------------|----------------|-----------|
| **0‑12 h** | • Deploy **all 5 buses** and **all 10 ambulances** to start loading the shelter.<br>• Load **all 150 in‑patients** (including 20 ICU) into ambul

## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">The model called <b>llama3.3</b> is FAR too large for home computers - it's not intended for personal computing and will consume all your resources! Stick with the nicely sized <b>llama3.2</b> or <b>llama3.2:1b</b> and if you want larger, try llama3.1 or smaller variants of Qwen, Gemma, Phi or DeepSeek. See the <A href="https://ollama.com/models">the Ollama models page</a> for a full list of models and sizes.
            </span>
        </td>
    </tr>
</table>

In [ ]:
!ollama pull llama3.2

In [13]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "gemma3:1b"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Okay, here’s a prioritized 72-hour operational plan for the island, aiming to minimize fatalities while considering the constraints and potential for catastrophic failure. This is a highly complex scenario, and this plan represents a *best-effort* approach, acknowledging significant risks. Ethical considerations are a core driver, prioritizing the most vulnerable.

**1. Assumptions and Objective Function**

*   **Assumptions:**
    *   Volcanic eruption is predicted within 72 hours, with a 30% chance within 48 hours and a 20% chance of delay to 96 hours.
    *   Current infrastructure (buses, ambulances, shelter, dam) is reasonably functional, albeit strained.
    *   Local population is largely stable, but a significant portion is elderly and vulnerable.
    *   Social media is rapidly disseminating misinformation, amplifying panic.
    *   Resource availability is realistic – we have sufficient fuel, shelter, and medical supplies.
    *   The risk of the dam failure is *relatively* low based on current conditions, but a worst-case scenario must be addressed.
*   **Objective Function:** Minimize *expected deaths* across all groups, weighted by vulnerability, while minimizing *system collapse* risk and *social unrest*. This is a combination of:
    *   **Mortality Reduction:** Directly minimizing deaths through evacuation and resource allocation.
    *   **System Collapse Risk:**  Reducing the risk of damage to infrastructure, resource shortages (fuel, medical supplies), and societal breakdown.
    *   **Social Unrest Mitigation:** Focusing on protecting vulnerable populations (elderly, children, and those with chronic conditions) to prevent escalation.

**2. Prioritized Operational Plan (72-Hour Timeline)**

| **Phase**           | **Time (Hours)** | **Actions**                                    | **Transport**                                 | **Shelter**                               | **Focus**                          | **Rationale**                                                               |
|----------------------|------------------|-------------------------------------------------|------------------------------------------------|-------------------------------------------|-------------------------------------|---------------------------------------------------------------------------------|
| **Phase 1: Immediate Response (Hours 0-24)** | 0-24              | **Evacuate Designated Zone:** Initial evacuation of vulnerable populations (elderly, children, IPs)  - 50 buses, 10 ambulances. | Military Airstrip, Buses (Priority Route), Ambulance (Secondary)| 3km Inland Shelter - 3,000 people, 2 weeks supplies | Prioritize evacuation of children and elderly, establish perimeter. |  Quickly remove a high-risk population to a safe zone. |
|                    |                    | **Immediate Stabilization:**  Coordinate with local authorities, provide basic first aid, address immediate threats. |  Ambulances (Emergency), Bus (Traffic Control) | 3km Inland Shelter - 3,000 people, 2 weeks supplies |  Maintain calm, address immediate medical emergencies | Reduces immediate mortality risk from evacuation. |
|                    |                    | **Dam Monitoring:**  Real-time visual inspection of dam integrity (if possible).   |  Military Airstrip,  Buses (Check Points) | 3km Inland Shelter - 3,000 people, 2 weeks supplies | Assess dam stability.             |  Prevent catastrophic failure by early detection.                         |
| **Phase 2: Consolidation & Resource Management (Hours 24-48)** | 24-48              | **Fuel & Supply Consolidation:**  Prioritize fuel and medical supplies for the remaining buses and ambulances.  | Military Airstrip, Buses (Fuel), Ambulance (Medical) | 3km Inland Shelter - 3,000 people, 2 weeks supplies | Secure critical supply chains.| Ensure immediate fuel needs are met.                         |
|                    |                    | **Refine Shelter:** Adjust shelter conditions based on initial observations and potential conditions | Military Airstrip, Buses (Personnel), Ambulance (Medical) | 3km Inland Shelter - 3,000 people, 2 weeks supplies |  Increase shelter capacity, secure perimeter. |  Reinforce shelter to avoid potential floods or deterioration from internal stress. |
|                    |                    | **Communications:** Establish secure communication channels (satellite, radio) - Military Team. | Military Team | 3km Inland Shelter - 3,000 people, 2 weeks supplies |  Centralized monitoring and data processing. |  Establish a hub for rapid information dissemination.                   |
| **Phase 3: Risk Assessment & Planning (Hours 48-72)** | 48-72              | **Volcano Threat Assessment:**  Detailed monitoring of volcanic activity (gas plume patterns, temperature, seismicity). | Military Airstrip, Buses (Monitoring), Ambulance (Data Collection) | 3km Inland Shelter - 3,000 people, 2 weeks supplies |  Determine best evacuation route.   |  Adapt plan based on evolving volcanic threat.                      |
|                    |                    | **Dam Monitoring Intensity:**  Increased monitoring and potential early warning system activation - Military Team | Military Airstrip, Buses (Monitoring), Ambulance (Coordination) | 3km Inland Shelter - 3,000 people, 2 weeks supplies |  Evaluate potential dam failure impact. |  Update dam design based on observed instability.                       |

**3. Trade-offs and Expected Outcomes**

*   **Best-Case Scenario (Volcano delays):**  The eruption is delayed to 96 hours.  We mitigate evacuation but suffer increased internal unrest due to the prolonged delay.
*   **Worst-Case Scenario (Volcano Triggered Immediately):**  The eruption begins within 48 hours.  We evacuate a large portion of the population, potentially causing massive casualties and social collapse.  Recovery takes longer.
*   **Moderate Scenario (Volcano with Significant Impact):**  The eruption triggers a significant avalanche, flooding lowlands, and significantly increases the risk of the dam failing in 48 hours, potentially causing substantial loss of life & devastation.  We need immediate action on storm control.

**4. Failure Modes and Mitigation**

1.  **Operational Failure (Chaos):**  Communication breakdown, insufficient personnel, poor logistical coordination. **Mitigation:**  Establish a clear chain of command, prioritize communication, maintain contact with local authorities.
2.  **Social Failure (Panic and Violence):**  Widespread panic leads to looting, violence, and breakdown of law enforcement.  **Mitigation:** Maintain calm.  Provide basic first aid.  Designate a buffer zone for non-combatants. Focus on containing the immediate crisis.
3.  **Informational Failure (Misinformation):**  The public is misled about the situation, leading to incorrect actions and increased fear.  **Mitigation:** Rapidly disseminate verified information through multiple channels - utilizing reliable sources (military, medical personnel, local authorities). Counter misinformation proactively.

**5.  Updated Plan & Data Thresholds**

*   *Volcanic Activity Increasing – 24 Hours:* Immediately increase alert level.  Begin deploying additional resources (medical, support personnel). Deploy sensor team to analyze data flow
*   *Dam Instability Detected - 24 Hours:*  Immediately alert military; initiate emergency stabilization procedures – if feasible.
*   *Evacuation Route Change Needed – 24 Hours:*  Re-evaluate evacuation routes based on real-time volcanic monitoring data.
*   *Increased Risk of Humanitarian Crisis - 48 Hours:*  Expand the shelter capacity and establish emergency medical teams to rapidly address needs.

**Ethical & Legal Considerations:**

*   **Prioritization of Vulnerable:**  The plan *must* prioritize the most vulnerable individuals (children, elderly, and those with chronic conditions) during all phases.
*   **Transparency:**  Communicate openly with the public about the situation and the plan’s objectives.
*   **Respect for Local Laws:**  While prohibiting nationality prioritization, the objective is to minimize harm to all. Respect existing local laws and regulations.
*   **Transparency of Resources:**  Justify resource allocation with clear metrics of benefit.

This plan balances immediate survival with long-term recovery, acknowledging the inherent uncertainty of the situation. It's a dynamic plan requiring continuous adaptation based on evolving information.  It will be crucial to constantly monitor the volcanic activity and adjust the plan accordingly.

In [14]:
# So where are we?

print(competitors)
print(answers)


['gpt-4.1-mini', 'claude-sonnet-4-5', 'gemini-2.5-flash', 'openai/gpt-oss-120b', 'gemma3:1b']
['**Context Summary:**  \nVolcano expected to erupt within 72h (30% chance <48h, 50% chance 48-72h, 20% chance 72-96h). Dam downstream likely to fail (60% chance within 48h if unattended), flooding lowlands and killing ~2,000. Population 10,000 with vulnerable groups and infrastructure constraints. Limited transport, shelter, and medical resources. Social/legal constraints prohibit nationality-based prioritization, demand protection of elderly/children, and require guarding incarcerated population. Airstrip limited by trained operator availability.\n\n---\n\n## 1. Assumptions and Objective Function\n\n### Assumptions:\n- **Eruption timing probabilities:**  \n   - <48h: 30%  \n   - 48-72h: 50%  \n   - 72-96h: 20%  \n- **Dam breach probability if unattended:** 60% within 48h, causing ~2,000 deaths. Dam can only be maintained by trained workers.  \n- **Evacuation transport capacities fixed:**  \n

In [15]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


Competitor: gpt-4.1-mini

**Context Summary:**  
Volcano expected to erupt within 72h (30% chance <48h, 50% chance 48-72h, 20% chance 72-96h). Dam downstream likely to fail (60% chance within 48h if unattended), flooding lowlands and killing ~2,000. Population 10,000 with vulnerable groups and infrastructure constraints. Limited transport, shelter, and medical resources. Social/legal constraints prohibit nationality-based prioritization, demand protection of elderly/children, and require guarding incarcerated population. Airstrip limited by trained operator availability.

---

## 1. Assumptions and Objective Function

### Assumptions:
- **Eruption timing probabilities:**  
   - <48h: 30%  
   - 48-72h: 50%  
   - 72-96h: 20%  
- **Dam breach probability if unattended:** 60% within 48h, causing ~2,000 deaths. Dam can only be maintained by trained workers.  
- **Evacuation transport capacities fixed:**  
   - Buses: 5 × 50 = 250 seats; 2 round trips → 500 seats total  
   - Ambulances: 1

In [16]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [17]:
print(together)

# Response from competitor 1

**Context Summary:**  
Volcano expected to erupt within 72h (30% chance <48h, 50% chance 48-72h, 20% chance 72-96h). Dam downstream likely to fail (60% chance within 48h if unattended), flooding lowlands and killing ~2,000. Population 10,000 with vulnerable groups and infrastructure constraints. Limited transport, shelter, and medical resources. Social/legal constraints prohibit nationality-based prioritization, demand protection of elderly/children, and require guarding incarcerated population. Airstrip limited by trained operator availability.

---

## 1. Assumptions and Objective Function

### Assumptions:
- **Eruption timing probabilities:**  
   - <48h: 30%  
   - 48-72h: 50%  
   - 72-96h: 20%  
- **Dam breach probability if unattended:** 60% within 48h, causing ~2,000 deaths. Dam can only be maintained by trained workers.  
- **Evacuation transport capacities fixed:**  
   - Buses: 5 × 50 = 250 seats; 2 round trips → 500 seats total  
   - Ambulance

In [18]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [19]:
print(judge)

You are judging a competition between 5 competitors.
Each model has been given this question:

You are the sole decision-maker responsible for the immediate crisis response on an island of 10,000 inhabitants where a volcano is forecast to erupt within 72 hours (with a 30% chance of occurring within 48 hours and a 20% chance of being delayed to 96 hours). You have the following resources and constraints:
- Transportation: 5 buses (50 seats each), 10 ambulances (4 seats each); fuel sufficient for two round trips of the buses and four round trips of the ambulances.
- Shelter and medical: a temporary shelter 3 km inland that holds 3,000 people with two weeks of basic supplies; a hospital with 200 beds and limited ICU; field medical teams that can expand capacity by 50 beds but will take 18 hours to set up; a dam downstream that, if unattended, has a 60% chance of failing within 48 hours and would flood lowlands killing ~2,000 people.
- Evacuation infrastructure: a military airstrip that ca

In [20]:
judge_messages = [{"role": "user", "content": judge}]

In [21]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


{"results": ["3", "1", "4", "2", "5"]}


In [22]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

Rank 1: gemini-2.5-flash
Rank 2: gpt-4.1-mini
Rank 3: openai/gpt-oss-120b
Rank 4: claude-sonnet-4-5
Rank 5: gemma3:1b


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>